# Fraude-360 — Pipeline local, étape par étape

Ce notebook exécute le pipeline complet sur le poste local, **une étape par cellule**, avec
affichage des données et des artefacts produits à chaque étape :

| # | Étape | Module |
|---|-------|--------|
| 00 | Génération / ingestion des données | `generation.simulateur` |
| 01 | Chargement Bronze (copie fidèle) | `ingestion.bronze` |
| 02 | Nettoyage + contrôles qualité → Silver | `quality.silver` |
| 03 | Ingénierie des variables → Gold | `features.gold` |
| 04 | Analyse exploratoire (description) | `eda.exploration` |
| 05 | Modélisation, entraînement, calibration, seuil $ | `modeling.entrainement` |
| 06 | Enregistrement MLflow + alias champion | `modeling.registre` |
| 07 | (option) A/B, scoring lot, surveillance, export BI | divers |

**Prérequis** : `pip install -r requirements.txt` + Java 17/21 (PySpark).  
Exécuter les cellules dans l'ordre (`Run All`).

## Préparation de l'environnement

In [ ]:
import json, os, sys, shutil
from pathlib import Path

# Racine du projet = dossier parent de notebooks/
RACINE = Path.cwd()
if RACINE.name == 'notebooks':
    RACINE = RACINE.parent
os.chdir(RACINE)
sys.path.insert(0, str(RACINE / 'src'))
print('Racine projet :', RACINE)
print('Interpréteur  :', sys.executable)   # doit pointer vers ...\frd\Scripts\python.exe

# Garde-fou : le kernel doit être celui du venv du projet (PySpark installé).
try:
    import pyspark  # noqa: F401
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "PySpark absent de ce kernel. Sélectionnez le kernel du venv 'frd' "
        "(voir README du notebook) puis relancez."
    ) from e

import pandas as pd
from IPython.display import Image, display
from fraude360.config import charger_config

# Jeu réduit (~3 min). Mettre CONFIG=None pour le jeu complet (15-30 min).
CONFIG = 'conf/config_rapide.yaml'
cfg = charger_config(CONFIG, forcer=bool(CONFIG)) if CONFIG else charger_config()
ART = Path(cfg['chemins']['artefacts'])

# --- Source des données -------------------------------------------------------
# DONNEES_LOCALES : dossier de VOS fichiers CSV de transactions déjà sur le PC.
#   None    -> on utilise data/00_landing tel quel ; un jeu synthétique n'est
#              généré QUE si ce dossier est vide.
#   chemin  -> ces CSV sont copiés dans data/00_landing avant l'ingestion.
# Le schéma attendu (en-têtes) :
#   transaction_id, horodatage (YYYY-MM-DD HH:MM:SS), carte_id, client_id,
#   marchand_id, categorie_marchand, canal, mode_saisie, montant, devise, pays,
#   latitude, longitude, authentification_3ds, appareil_id, adresse_ip,
#   duree_session_s, est_fraude (0/1), typologie_fraude, horodatage_label
DONNEES_LOCALES = None   # ex. r"D:\donnees\transactions"

def montrer_json(chemin):
    chemin = Path(chemin)
    print(json.dumps(json.loads(chemin.read_text('utf-8')), indent=2, ensure_ascii=False))

def montrer_figures(dossier, motif='*.png'):
    for p in sorted(Path(dossier).glob(motif)):
        print(p.name); display(Image(str(p)))

print('Config chargée :', cfg['projet']['nom'], '| environnement =', cfg['environnement']['cible'])

## Étape 00 — Données sources (locales ou synthétiques)

La zone d'atterrissage est `data/00_landing/`. Le pipeline ingère **tous les `*.csv`**
qui s'y trouvent.

- **Utiliser vos données** : renseignez `DONNEES_LOCALES` dans la cellule de préparation
  (dossier de vos CSV) — ils seront copiés ici. Sinon, déposez-les vous-même dans
  `data/00_landing/`.
- **Sinon** : si le dossier est vide, un flux transactionnel synthétique réaliste
  (plusieurs typologies de fraude) est généré automatiquement.

In [ ]:
landing = Path(cfg['chemins']['atterrissage'])
landing.mkdir(parents=True, exist_ok=True)

# 1) Copier vos CSV locaux dans la zone d'atterrissage, si un dossier est fourni
if DONNEES_LOCALES:
    src = Path(DONNEES_LOCALES)
    csv_src = sorted(src.glob('*.csv'))
    if not csv_src:
        raise FileNotFoundError(f'Aucun .csv dans {src}')
    for f in csv_src:
        shutil.copy2(f, landing / f.name)
    print(f'{len(csv_src)} fichier(s) copié(s) depuis {src}')

# 2) Générer un jeu synthétique seulement si la zone est vide
fichiers = sorted(landing.glob('*.csv'))
if not fichiers:
    print('Zone d\'atterrissage vide -> génération d\'un jeu synthétique...')
    from fraude360.generation import simulateur
    simulateur.executer(cfg)
    fichiers = sorted(landing.glob('*.csv'))
else:
    print(f'{len(fichiers)} fichier(s) déjà présents dans {landing} -> pas de génération.')

# 3) Aperçu des données sources
df0 = pd.concat([pd.read_csv(f) for f in fichiers], ignore_index=True)
print('Transactions :', len(df0), '| Colonnes :', len(df0.columns))
if 'est_fraude' in df0.columns:
    n = pd.to_numeric(df0['est_fraude'], errors='coerce').fillna(0)
    print('Fraudes :', int(n.sum()), f'({n.mean():.2%})')
df0.head()

## Étape 01 — Chargement Bronze

Ingestion incrémentale et idempotente des fichiers de `landing` vers `data/01_bronze/`
(copie fidèle + métadonnées techniques, schéma contractuel).

In [ ]:
from fraude360.ingestion import bronze
from fraude360.utils.spark import obtenir_spark

chemin_bronze = bronze.executer(cfg, complet=True)
spark = obtenir_spark(cfg)
b = spark.read.parquet(str(chemin_bronze))
print('Bronze :', b.count(), 'lignes,', len(b.columns), 'colonnes')
b.limit(5).toPandas()

## Étape 02 — Nettoyage et contrôles qualité → Silver

7 contrôles (colonnes obligatoires, bornes de montant, devises, doublons…), quarantaine
tracée des lignes rejetées, échec du lot si le taux de rejet dépasse le seuil bloquant.

In [ ]:
from fraude360.quality import silver
rapport_q = silver.executer(cfg)
montrer_json(ART / 'qualite' / 'rapport_qualite.json')

## Étape 03 — Ingénierie des variables → Gold

~44 variables en fenêtres glissantes par carte / client / marchand (vélocité, distance,
écarts aux habitudes…), calculées sans fuite temporelle.

In [ ]:
from fraude360.features import gold
from fraude360.eda.exploration import charger_gold_pandas

gold.executer(cfg)
g = charger_gold_pandas(cfg)
print('Gold :', g.shape[0], 'lignes ×', g.shape[1], 'colonnes')
print('Cible est_fraude :', int(g['est_fraude'].sum()), f"({g['est_fraude'].mean():.2%})")
print('\nColonnes :', list(g.columns))

### Description statistique des variables Gold

In [ ]:
with pd.option_context('display.max_columns', None, 'display.width', 200):
    display(g.describe(include='number').T)
    display(g.describe(include='object').T)

## Étape 04 — Analyse exploratoire

8 figures + synthèse : séries temporelles, typologies actives, montants, heures, canaux,
pouvoir discriminant des variables, corrélations, vélocité.

In [ ]:
from fraude360.eda import exploration
exploration.executer(cfg)
montrer_json(ART / 'eda' / 'synthese_eda.json')
montrer_figures(ART / 'eda')

## Étape 05 — Modélisation, entraînement, calibration, seuil en dollars

Découpage **temporel** (train / valid / test), comparaison de 3 modèles sur la PR-AUC,
calibration isotonique des probabilités, puis choix du **seuil de décision par minimisation
du coût en dollars** sous contrainte du budget quotidien d'alertes. Suivi complet dans MLflow.

In [ ]:
from fraude360.modeling import entrainement
entrainement.executer(cfg, etiquette='v1')

resume = json.loads((ART / 'modele' / 'resume_courant.json').read_text('utf-8'))
print('Modèle retenu :', resume['modele_retenu'])
print('Seuil de décision ($) :', round(resume['seuil_decision'], 4))
print('PR-AUC test :', round(resume['metriques_test']['pr_auc'], 4),
      '| Rappel :', round(resume['metriques_test']['rappel'], 4),
      '| Précision :', round(resume['metriques_test']['precision'], 4))
print('Économie vs sans modèle ($) :', resume.get('economie_vs_sans_modele'))

In [ ]:
# Importance des variables + courbe coût / seuil + figures
imp = ART / 'modele' / 'importances_v1.csv'
if imp.exists():
    display(pd.read_csv(imp).head(15))
montrer_figures(ART / 'modele')

## Étape 06 — Enregistrement MLflow et alias champion

Le modèle calibré est journalisé dans le registre MLflow local (`mlruns/`) et reçoit
l'alias `champion` : c'est cet objet exact que servira la production (API et scoring par lots).

In [ ]:
import mlflow
from fraude360.modeling import registre

client = registre.configurer(cfg)
v_champ = registre.version_par_alias(registre.ALIAS_CHAMPION, cfg)
print('Version champion :', v_champ)

nom_mod = cfg['databricks']['modele_enregistre'] if cfg.sur_databricks else registre.NOM_MODELE
try:
    for mv in client.search_model_versions(f"name='{nom_mod}'"):
        print(f"  v{mv.version:<3} run={mv.run_id[:8]}  alias={list(mv.aliases)}")
except Exception as e:
    print('(registre)', e)

df_runs = mlflow.search_runs(search_all_experiments=True,
                             order_by=['start_time DESC'], max_results=5)
cols = [c for c in ['run_id', 'metrics.test_pr_auc', 'params.seuil_decision',
                    'metrics.economie_dollars_vs_sans_modele'] if c in df_runs.columns]
df_runs[cols]

In [ ]:
# Recharger le champion et scorer 5 transactions Gold pour vérifier l'objet servi
modele = registre.charger_par_alias(registre.ALIAS_CHAMPION, cfg)
cible = cfg['modelisation']['cible']
X = g.drop(columns=[c for c in [cible, 'transaction_id', 'horodatage'] if c in g.columns])
X = X.select_dtypes('number').head(5)
print('Scores champion :', modele.predict_proba(X)[:, 1].round(4))

## Étape 07 — (option) Suite du pipeline

Décommenter pour enchaîner : test A/B règles vs modèle, scoring par lots (table
`gold/decisions`), surveillance (PSI / KS / dérive), puis exports Power BI.

In [ ]:
# from fraude360.abtest import experience;      experience.executer(cfg)
# from fraude360.serving import lot;            lot.executer(cfg)
# from fraude360.monitoring import surveillance; surveillance.executer(cfg)
# from fraude360.bi import export;              export.executer(cfg)
# montrer_json(ART / 'surveillance' / 'rapport_surveillance.json')

---
**Terminé.** Artefacts dans `artifacts/`, runs MLflow dans `mlruns/`.  
Équivalent une commande : `python -m fraude360 tout --config conf/config_rapide.yaml`.